# Results Analysis

Run after `scripts/03_evaluate.py --split test`. Requires `results/test_per_case.csv` and `results/test_summary.json`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

RESULTS = '../results'
df = pd.read_csv(f'{RESULTS}/test_per_case.csv')
with open(f'{RESULTS}/test_summary.json') as f:
    summary = json.load(f)
print(f'{len(df)} test cases')
print(df.describe().T.round(3))

In [ ]:
# Overall results table with CIs
rows = []
for name, s in summary['overall'].items():
    rows.append({
        'Structure': name,
        'Dice mean': f"{s['dice']['mean']:.3f}",
        'Dice 95% CI': f"[{s['dice']['ci_lo']:.3f}, {s['dice']['ci_hi']:.3f}]",
        'HD95 mean (mm)': f"{s['hd95']['mean']:.2f}",
        'HD95 95% CI': f"[{s['hd95']['ci_lo']:.2f}, {s['hd95']['ci_hi']:.2f}]",
    })
pd.DataFrame(rows)

In [ ]:
# Dice distributions per structure — violin + strip
structures = ['lv_cavity', 'myocardium', 'left_atrium']
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, name in zip(axes, structures):
    vals = df[f'dice_{name}'].dropna()
    ax.violinplot(vals, positions=[0], showmedians=True)
    ax.set_xticks([])
    ax.set_title(name.replace('_',' '))
    ax.set_ylim(0, 1)
    ax.set_ylabel('Dice' if name == structures[0] else '')
    ax.axhline(vals.mean(), color='red', linestyle='--', alpha=0.5, label=f'mean={vals.mean():.3f}')
    ax.legend(fontsize=8)

plt.suptitle('Dice score distributions on test set')
plt.tight_layout()
plt.show()

In [ ]:
# Stratified Dice by quality — interactive plotly
melted = df.melt(id_vars=['patient_id','view','phase','quality'],
                  value_vars=[f'dice_{n}' for n in structures],
                  var_name='structure', value_name='dice')
melted['structure'] = melted['structure'].str.replace('dice_','').str.replace('_',' ')

fig = px.box(melted, x='quality', y='dice', color='structure',
             facet_col='structure', title='Dice by image quality')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Worst cases — failure gallery
for name in structures:
    worst = df.nsmallest(5, f'dice_{name}')
    print(f'\n--- Worst 5 for {name} ---')
    print(worst[['patient_id','view','phase','quality',f'dice_{name}',f'hd95_{name}']].to_string(index=False))